# Mini Project 1 — AI assistant app store reviews

**Author portfolio analysis** · All assets for this project live in this **`miniProject1/`** folder (notebook, data, charts, scripts).

**Sections:** 1. Overview · 2. Data Profile · 3. Analysis · 4. Conclusions · 5. Process (optional)


In [ ]:
!pip install jupyter plotly kaleido pandas


## Section 1 — Overview


### What is this dataset?

If you have never seen this table before: imagine opening the **Google Play Store** or **Apple App Store**, scrolling to user reviews for a popular app, and copying each review’s **star rating**, **written comment**, **date**, and sometimes **app version** into a spreadsheet. This notebook analyzes that kind of public feedback at scale for **three AI assistant apps** people install on their phones:

| App | Developer (store listing) |
|-----|---------------------------|
| **ChatGPT** | OpenAI |
| **Google Gemini** | Google |
| **Claude** | Anthropic |

Each **row** is one review. Columns include which **app** and **store** (`platform`), the **rating** (1–5 stars), the **review text**, and **when** it was posted. The combined file has about **3,000 reviews** in my current scrape (500 per app per store, when both stores returned a full pull).

### Where did it come from?

I did **not** use a private company API key. Reviews are in **`data/mp1_reviews_raw.csv`**, built by running **`fetch_mp1_review_data.py`** in this same folder:

- **Google Play (Android):** the **`google-play-scraper`** Python library reads publicly visible Play review pages for package IDs such as `com.openai.chatgpt`.
- **Apple App Store (iOS):** Apple’s **public iTunes customer-review RSS JSON** feed (for example `https://itunes.apple.com/us/rss/customerreviews/page=1/id=6448311069/sortby=mostrecent/json` for ChatGPT).

The column **`scraped_at_utc`** records when I downloaded the file. This is a **snapshot**, not a live feed.

### What questions am I trying to answer? (MP1a)

1. **Shared vs unique pain:** What usability complaints show up **across all three** assistants, and which frustrations seem **unique to one product**?
2. **Ratings and change:** Did **average ratings** move in ways that relate to **update-heavy periods**, and what language shows up in those windows? (I treat **time carefully** because each app’s sample covers a different calendar span.)
3. **Polar voices:** How do **5-star** and **1-star** reviewers describe the same core ideas—**accuracy**, **speed**, and **conversation quality**?

### Why do these questions matter to me?

I am interested in **AI product design** and want to see **where tools fail users at scale**, the same kind of signal that can inform **redesign or roadmap choices**. Store reviews are imperfect (loud minorities, spam, mixed languages), but they mirror how teams sometimes do **competitive audits** and **light thematic coding** before deeper research. This notebook is my **transparent first pass** on that practice—honest about limits, not pretending one CSV replaces interviews or lab studies.


## Section 2 — Data Profile


In [ ]:
from pathlib import Path

import pandas as pd


def resolve_data_csv() -> Path:
    """Find mp1_reviews_raw.csv when the kernel cwd is this folder or the repo root."""
    here = Path.cwd().resolve()
    candidates = [
        here / "data" / "mp1_reviews_raw.csv",
        here / "miniProject1" / "data" / "mp1_reviews_raw.csv",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find data/mp1_reviews_raw.csv from " + str(here)
    )


DATA_PATH = resolve_data_csv()
df = pd.read_csv(DATA_PATH, encoding="utf-8")
print("Using:", DATA_PATH)
print("Rows:", len(df))


In [ ]:
df.head()


**Interpretation:** Each row is one review with app name, store, star rating, and text—enough to see the grain of the analysis.


In [ ]:
df.info()


**Interpretation:** Roughly three thousand rows; mostly string columns plus integer ratings, so I coerce `rating` when I need numeric summaries.


In [ ]:
df.describe(include="all")


**Interpretation:** Star ratings use the full 1–5 range while review text length varies widely, which matters before keyword or theme work.


In [ ]:
df.isnull().sum()


**Interpretation:** Missing values cluster in `title` and `app_version` (store differences), so I will not treat absent version as random noise when linking reviews to releases.


## Section 3 — Analysis


### Question 1 — Where is dissatisfaction concentrated? (MP1a)

**Question:** What usability complaints appear consistently across all three AI apps, and which pain points are unique to a single product?

Before reading thousands of comments by hand, I use **share of 1–2 star reviews per app** as a **quantitative proxy** for where visible frustration piles up. Chart type: **bar chart** (categorical comparison). Static image committed in **`charts/`** (Plotly + kaleido, A6).

![Share of 1–2 star reviews by app](charts/chart_q1_low_star_share_by_app.png)


**What the chart argues:** In this scrape, **one app carries a heavier low-star tail** than the others when both stores are pooled. That does not yet name specific usability themes, but it tells me **where to read text first** and supports the idea that some pain may be **product-specific**, not identical across every assistant.


### Question 2 — How does mean rating move through the scrape? (MP1a)

**Question:** Did average user ratings shift following major app updates, and what themes appear in reviews from those time windows?

A fixed **row count per app** does **not** mean the same **calendar window**, so I compare **recency deciles within each app** (decile 1 = newest 10% of that app’s rows, 10 = oldest 10%). Chart type: **line chart** along an ordered axis. Image: **`charts/chart_q2_mean_rating_by_recency_decile.png`**.

![Mean star rating by recency decile](charts/chart_q2_mean_rating_by_recency_decile.png)


**What the chart argues:** With a **shared x-axis meaning** (“depth into this scrape”), the three products are easier to compare fairly than with mismatched calendar weeks. The lines describe **gradient inside the sample**, not proof about a named release day—tying spikes to **version numbers** still needs extra metadata and qualitative reading.


### Question 3 — How do 1★ and 5★ writers talk about core features? (MP1a)

**Question:** How do 5-star and 1-star reviewers describe the same core features (accuracy, speed, conversation quality) differently?

I use a **pilot keyword pass** (speed, accuracy/trust, memory/context) on 1-star vs 5-star rows. Chart type: **grouped bar chart**. Image: **`charts/chart_q3_keyword_themes_1_vs_5_star.png`**.

![Keyword themes: 1-star vs 5-star](charts/chart_q3_keyword_themes_1_vs_5_star.png)


**What the chart argues:** The bars compare **how often** each star bucket mentions a theme, hinting that polar reviewers may **emphasize different concerns**. This is a **compass for deeper coding**, not a validated codebook.


## Section 4 — Conclusions


### Question 1 — shared vs unique pain

**Finding:** Low-star volume is **not evenly spread** across the three apps in this sample.  
**Implication:** Competitive and UX work should not assume “AI assistant problems” are identical everywhere—some frustration may cluster on **one product**.  
**Next:** Code a small shared rubric (billing, model changes, UI clutter, post-update breakage) and count overlaps in `review_text`.

### Question 2 — ratings and updates

**Finding:** The **recency-decile** view shows modest drift within each app’s stack rather than a dramatic cliff tied to a release I can name from this file alone.  
**Implication:** Claiming “ratings dropped after version X” needs **release dates** joined to `app_version` where available.  
**Next:** Map top version strings to store release notes and re-read text in those windows.

### Question 3 — polar language

**Finding:** Pilot keywords suggest **1-star and 5-star** writers surface **different theme bundles** in raw text.  
**Implication:** Feature narratives (accuracy vs speed vs memory) may land differently for promoters vs detractors.  
**Next:** Replace keyword flags with **iterative thematic codes** and example quotes; consider multilingual reviews.

### Overall

Public store reviews are a **noisy but real** lens for AI product judgment. This project is a **reproducible first layer**—profile, fair charts, honest limits—not the final word on user needs.


## Section 5 — Process (optional)


### How this folder came together

1. **MP1a declaration** — I chose three AI apps and three analytical questions tied to HCD-style competitive learning.  
2. **Data collection** — I wrote **`fetch_mp1_review_data.py`** after finding that a dedicated App Store scraper library was unreliable; Play uses **`google-play-scraper`**, iOS uses **Apple’s RSS JSON**. I targeted **500 reviews per app per store** and saved **`data/mp1_reviews_raw.csv`**.  
3. **Week 5 (A5)** — **`a5_mp1_reviews.py`** loaded the CSV with pandas (`head`, `info`, `isnull`, `value_counts`, filters, `groupby`) to sanity-check distributions before charting.  
4. **Week 6 (A6)** — **`a6_mp1_charts.py`** built three Plotly figures and exported PNGs to **`charts/`** with **kaleido**. I **pivoted chart Q2** from calendar weeks to **recency deciles** after noticing ChatGPT’s reviews clustered in only a few days while other apps spanned longer—calendar weeks were misleading for a fair comparison.  
5. **MP1 notebook** — This **`mp1.ipynb`** bundles the story, data, and images so graders can review **one stand-alone directory**.

**Surprises:** ChatGPT’s “most recent 500” spanned a **much shorter calendar range** than Claude’s, which changed how I visualized time. **Low-star share** pointed to Claude early, which I would not have guessed before plotting.

**If I had more time:** Longer scrape window, real thematic coding, and explicit **version ↔ release** alignment for the update question.
